# Exercise: Configuration File Processing Pipeline

You are a DevOps engineer responsible for building a **memory-efficient configuration file parser**.

The configuration files can be very large, so loading the entire file into memory is not an option. Instead, you must process the file **lazily** using a pipeline of **generator functions**.

---

# Configuration File Format

The configuration file follows these rules:

### Sections

A section is defined using square brackets:

```text
[database]
[api_service]
```

---

### Key-Value Pairs

A configuration entry is written as:

```text
key = value
```

Examples:

```text
host = localhost
port = 5432
timeout = 30
```

---

### Comments

Any line that starts with `#` is a comment and should be ignored.

Example:

```text
# Database configuration
```

---

### Empty Lines

Blank lines or lines containing only whitespace should also be ignored.

---

# Objective

Build a pipeline consisting of **three generator functions**.

---

# Function 1: `read_config_file(filepath)`

### Signature

```python
def read_config_file(filepath):
    pass
```

### Responsibilities

- Accept a file path as a string.
- Open the file.
- Read it **line by line**.
- Yield each line exactly as it appears in the file.

---

# Function 2: `filter_config_lines(lines)`

### Signature

```python
def filter_config_lines(lines):
    pass
```

### Responsibilities

- Accept an iterable of lines (such as the output of `read_config_file`).
- Remove leading and trailing whitespace using `strip()`.
- Ignore:
  - Empty lines
  - Comment lines (starting with `#`)
- Yield only valid, cleaned lines.

---

# Function 3: `parse_config_lines(lines)`

### Signature

```python
def parse_config_lines(lines):
    pass
```

### Responsibilities

- Accept an iterable of clean lines (such as the output of `filter_config_lines`).
- Keep track of the **current section**.

The initial section should be:

```python
None
```

because key-value pairs may appear before any section header.

---

## When a Section is Found

Example:

```text
[database]
```

Update the current section:

```python
current_section = "database"
```

Do **not** yield anything.

---

## When a Key-Value Pair is Found

Example:

```text
host = localhost
```

Yield:

```python
(current_section, "host", "localhost")
```

---

# Assumptions

For this exercise, you may assume:

- The file path is always valid.
- Every non-comment, non-empty line is either:
  - a section header, or
  - a valid key-value pair.

---

# Example

## File: `app.cfg`

```text
# Global settings
timeout = 30

[database]
host = db.prod.local
port = 5432

[api_service]
url = https://api.service.com/v1?retries=3
```

---

# Expected Output

```python
(None, 'timeout', '30')
('database', 'host', 'db.prod.local')
('database', 'port', '5432')
('api_service', 'url', 'https://api.service.com/v1?retries=3')
```

---

# Generator Pipeline

```python
lines = read_config_file("app.cfg")
clean_lines = filter_config_lines(lines)
configs = parse_config_lines(clean_lines)

for item in configs:
    print(item)
```

---

# How Your Solution Will Be Tested

Your pipeline will be tested with:

1. A standard configuration file containing multiple sections.
2. A file with key-value pairs defined before the first section.
3. A file where values contain an `=` character.
4. An empty file.

---

# Goal

This exercise is designed to test:

- Generator functions (`yield`)
- Generator pipelines
- File handling
- String processing (`strip`, `startswith`, `split`)
- Stateful parsing using generators
- Writing memory-efficient code for large configuration files

In [2]:
def read_config_file(filepath):
    """
    Validates the filepath and yields each line from the file.

    Args:
        filepath (str): The path to the configuration file.

    Yields:
        str: A single line from the file.
    """

    with open(filepath, "r") as f:
        for line in f:
            yield line


def filter_config_lines(lines):
    """
    Filters an iterable of lines, yielding stripped, non-empty, non-comment lines.

    Args:
        lines (iterable): An iterable producing string lines.

    Yields:
        str: A line that is not a comment or empty.
    """

    for line in lines:
        line = line.strip()
        if line and not line.startswith("#"):
            yield line


def parse_config_lines(lines):
    """
    Parses an iterable of clean config lines into (section, key, value) tuples.

    Args:
        lines (iterable): An iterable producing clean config lines.

    Yields:
        tuple: A tuple in the format (section, key, value).
    """
    current_section = None

    for line in lines:
        if line.startswith('[') and line.endswith(']'):
            current_section = line[1:-1]
        elif '=' in line:
            key, value = line.split('=', 1)
            yield current_section, key.strip(), value.strip()

lines = read_config_file("app.cnf")
clean_lines = filter_config_lines(lines)
configs = parse_config_lines(clean_lines)
for item in configs:
    print(item)

(None, 'timeout', '30')
(None, 'retry_count', '5')
('database', 'host', 'db.prod.local')
('database', 'port', '5432')
('database', 'username', 'admin')
('database', 'password', 'secret123')
('redis', 'host', 'redis.prod.local')
('redis', 'port', '6379')
('redis', 'db', '0')
('api_service', 'url', 'https://api.service.com/v1?retries=3')
('api_service', 'timeout', '60')
('api_service', 'api_key', 'abc123xyz')
('logging', 'level', 'INFO')
('logging', 'file', '/var/log/app.log')
('email', 'smtp_server', 'smtp.gmail.com')
('email', 'smtp_port', '587')
('email', 'sender', 'admin@example.com')
